<a href="https://colab.research.google.com/github/johnsimon3110-star/Project19_Order_Informed_Domain_Generalization/blob/main/Project19_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 19 — Order-Informed Domain Generalization for Rotating-Machinery Fault Diagnosis

This notebook trains three models from the existing prepared Paderborn windows and evaluates them on the four recording-level held-out operating-condition folds.

**Models:** Raw CNN, Order CNN, Order CNN with consistency objective.

**Important:** This notebook reuses the already extracted signal/window NPZ files and the existing Stage 5/Stage 7 manifests. It does not repeat MATLAB extraction or order conversion. It creates separate output folders and does not overwrite the previous experiment.

Run the cells from top to bottom. Training all three models across four folds can take time.


In [16]:

# ==========================================================
# CELL 1 — ENVIRONMENT, DRIVE, AND PROJECT PATHS
# Objective: initialize TensorFlow, connect Drive, and define
# separate output locations for this fresh training run.
# ==========================================================

import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import drive

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/BearingFaultResearch"
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RAW_WINDOWS_DIR = os.path.join(DATA_DIR, "raw_windows")
ORDER_WINDOWS_DIR = os.path.join(DATA_DIR, "order_windows")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

# Keep this run separate from the earlier experiment.
RUN_ROOT = os.path.join(PROJECT_ROOT, "fresh_train_predict_run")
RUN_RESULTS = os.path.join(RUN_ROOT, "results")
RUN_MODELS = os.path.join(RUN_ROOT, "models")
RUN_PREDICTIONS = os.path.join(RUN_ROOT, "predictions")

for folder in [RUN_ROOT, RUN_RESULTS, RUN_MODELS, RUN_PREDICTIONS]:
    os.makedirs(folder, exist_ok=True)

STAGE5_PATH = os.path.join(RESULTS_DIR, "stage5_condition_split_manifest.csv")
STAGE7_PATH = os.path.join(RESULTS_DIR, "stage7_order_window_manifest.csv")

print("=" * 70)
print("PROJECT 19 — FRESH TRAINING AND PREDICTION")
print("=" * 70)
print("Python:", __import__("sys").version)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Raw windows:", RAW_WINDOWS_DIR)
print("Order windows:", ORDER_WINDOWS_DIR)
print("Stage 5 manifest exists:", os.path.isfile(STAGE5_PATH))
print("Stage 7 manifest exists:", os.path.isfile(STAGE7_PATH))
print("New run folder:", RUN_ROOT)

assert os.path.isdir(RAW_WINDOWS_DIR), "Raw-window directory not found."
assert os.path.isdir(ORDER_WINDOWS_DIR), "Order-window directory not found."
assert os.path.isfile(STAGE5_PATH), "Stage 5 split manifest not found."
assert os.path.isfile(STAGE7_PATH), "Stage 7 order-window manifest not found."

print("\nCELL 1 COMPLETED")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT 19 — FRESH TRAINING AND PREDICTION
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
TensorFlow: 2.21.0
NumPy: 2.4.6
GPUs: []
Raw windows: /content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows
Order windows: /content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows
Stage 5 manifest exists: True
Stage 7 manifest exists: True
New run folder: /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run

CELL 1 COMPLETED


In [17]:

# ==========================================================
# CELL 2 — LOAD RECORDING SPLITS AND BUILD WINDOW MANIFESTS
# Objective: reuse the approved four-fold recording splits,
# then create one row per saved raw/order window.
#
# No recordings are re-split. No signals are re-extracted.
# ==========================================================

stage5 = pd.read_csv(STAGE5_PATH)
stage7 = pd.read_csv(STAGE7_PATH)

def find_column(df, candidates):
    lookup = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    raise KeyError(f"None of these columns found: {candidates}. Available: {list(df.columns)}")

s5_filename = find_column(stage5, ["filename"])
s5_bearing = find_column(stage5, ["bearing_id", "bearing_ID"])
s5_label = find_column(stage5, ["label"])
s5_fold = find_column(stage5, ["fold"])
s5_split = find_column(stage5, ["split"])
s5_condition = find_column(stage5, ["held_out_condition"])

s7_filename = find_column(stage7, ["filename"])
s7_bearing = find_column(stage7, ["bearing_id", "bearing_ID"])
s7_label = find_column(stage7, ["label"])
s7_count = find_column(stage7, ["number_of_windows"])
s7_path = find_column(stage7, ["output_path"])

# Canonical recording-level fold table.
fold_records = stage5[[s5_filename, s5_bearing, s5_label, s5_fold, s5_split, s5_condition]].copy()
fold_records.columns = ["filename", "bearing_id", "label", "fold", "split", "held_out_condition"]

fold_records["filename"] = fold_records["filename"].astype(str)
fold_records["bearing_id"] = fold_records["bearing_id"].astype(str)
fold_records["label"] = fold_records["label"].astype(int)
fold_records["fold"] = fold_records["fold"].astype(int)
fold_records["split"] = fold_records["split"].astype(str).str.lower()

# Each recording must have one assignment per fold.
assert not fold_records.duplicated(["fold", "filename"]).any(), "Duplicate recording assignment within a fold."
assert set(fold_records["fold"].unique()) == {1, 2, 3, 4}, "Expected folds 1, 2, 3, and 4."

# Build raw-window rows from the per-recording raw NPZ files.
raw_rows = []
missing_raw = []

for row in fold_records.itertuples(index=False):
    filename = row.filename
    stem = os.path.splitext(os.path.basename(filename))[0]
    npz_path = os.path.join(RAW_WINDOWS_DIR, stem + "_raw_windows.npz")

    if not os.path.isfile(npz_path):
        missing_raw.append(npz_path)
        continue

    with np.load(npz_path) as data:
        if "raw_windows" not in data:
            raise KeyError(f"'raw_windows' key missing in {npz_path}")
        n_windows = data["raw_windows"].shape[0]

    for window_index in range(n_windows):
        raw_rows.append({
            "filename": filename,
            "bearing_id": row.bearing_id,
            "label": int(row.label),
            "fold": int(row.fold),
            "split": row.split,
            "held_out_condition": row.held_out_condition,
            "window_index": window_index,
            "output_path": npz_path
        })

assert not missing_raw, (
    f"{len(missing_raw)} raw NPZ files are missing. Example: {missing_raw[:5]}"
)

raw_manifest = pd.DataFrame(raw_rows)

# Merge the Stage 7 order-window manifest with the exact same
# Stage 5 fold/split assignments, then expand each recording
# into its saved order-domain windows.
order_recordings = stage7[[s7_filename, s7_bearing, s7_label, s7_count, s7_path]].copy()
order_recordings.columns = ["filename", "bearing_id", "label", "number_of_windows", "output_path"]
order_recordings["filename"] = order_recordings["filename"].astype(str)
order_recordings["bearing_id"] = order_recordings["bearing_id"].astype(str)
order_recordings["label"] = order_recordings["label"].astype(int)
order_recordings["number_of_windows"] = order_recordings["number_of_windows"].astype(int)
order_recordings["output_path"] = order_recordings["output_path"].astype(str)

order_recordings = order_recordings.merge(
    fold_records,
    on=["filename", "bearing_id", "label"],
    how="inner",
    validate="one_to_many"
)

order_rows = []
missing_order = []

for row in order_recordings.itertuples(index=False):
    npz_path = row.output_path

    if not os.path.isfile(npz_path):
        missing_order.append(npz_path)
        continue

    with np.load(npz_path) as data:
        if "order_windows" not in data:
            raise KeyError(f"'order_windows' key missing in {npz_path}")
        windows = data["order_windows"]
        n_windows = windows.shape[0]

    if n_windows != int(row.number_of_windows):
        raise ValueError(
            f"Window count mismatch for {row.filename}: "
            f"manifest={row.number_of_windows}, NPZ={n_windows}"
        )

    for window_index in range(n_windows):
        order_rows.append({
            "filename": row.filename,
            "bearing_id": row.bearing_id,
            "label": int(row.label),
            "fold": int(row.fold),
            "split": row.split,
            "held_out_condition": row.held_out_condition,
            "window_index": window_index,
            "output_path": npz_path
        })

assert not missing_order, (
    f"{len(missing_order)} order NPZ files are missing. Example: {missing_order[:5]}"
)

order_manifest = pd.DataFrame(order_rows)

raw_manifest.to_csv(os.path.join(RUN_RESULTS, "fresh_raw_window_manifest.csv"), index=False)
order_manifest.to_csv(os.path.join(RUN_RESULTS, "fresh_order_window_manifest.csv"), index=False)

print("RECORDING SPLIT COUNTS")
print(fold_records.groupby(["fold", "split"]).agg(
    recordings=("filename", "nunique"),
    healthy=("label", lambda x: int((x == 0).sum())),
    damaged=("label", lambda x: int((x == 1).sum()))
))

print("\nWINDOW MANIFEST SIZES")
print("Raw window rows:", len(raw_manifest))
print("Order window rows:", len(order_manifest))
print("Unique raw recordings:", raw_manifest["filename"].nunique())
print("Unique order recordings:", order_manifest["filename"].nunique())

print("\nCELL 2 COMPLETED")


RECORDING SPLIT COUNTS
                 recordings  healthy  damaged
fold split                                   
1    test               120       60       60
     train              288      144      144
     validation          72       36       36
2    test               120       60       60
     train              288      144      144
     validation          72       36       36
3    test               120       60       60
     train              288      144      144
     validation          72       36       36
4    test               120       60       60
     train              288      144      144
     validation          72       36       36

WINDOW MANIFEST SIZES
Raw window rows: 39396
Order window rows: 39396
Unique raw recordings: 480
Unique order recordings: 480

CELL 2 COMPLETED


In [18]:

# ==========================================================
# CELL 3 — DISK-BACKED WINDOW SEQUENCES AND NORMALIZATION
# Objective: load windows in batches without loading the full
# dataset into RAM. Normalization statistics are calculated
# from training windows only, separately for each fold.
# ==========================================================

from tensorflow.keras.utils import Sequence

RAW_WINDOW_KEY = "raw_windows"
ORDER_WINDOW_KEY = "order_windows"

class WindowSequence(Sequence):
    def __init__(
        self, frame, window_key, window_size, mean, std,
        batch_size=16, shuffle=False, augment=False,
        return_labels=True, seed=42
    ):
        super().__init__()
        self.frame = frame.reset_index(drop=True).copy()
        self.window_key = window_key
        self.window_size = int(window_size)
        self.mean = float(mean)
        self.std = max(float(std), 1e-8)
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.augment = bool(augment)
        self.return_labels = bool(return_labels)
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(self.frame))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)

    def _read_window(self, row):
        with np.load(row.output_path) as data:
            window = data[self.window_key][int(row.window_index)].astype(np.float32)
        window = (window - self.mean) / self.std
        return window.reshape(self.window_size, 1)

    def __getitem__(self, batch_index):
        batch_ids = self.indices[
            batch_index * self.batch_size:(batch_index + 1) * self.batch_size
        ]
        batch = self.frame.iloc[batch_ids]

        x = np.empty((len(batch), self.window_size, 1), dtype=np.float32)
        y = np.empty((len(batch), 1), dtype=np.float32)

        for j, row in enumerate(batch.itertuples(index=False)):
            x[j] = self._read_window(row)
            y[j, 0] = float(row.label)

        if self.augment:
            scale = self.rng.uniform(0.9, 1.1, size=(len(x), 1, 1)).astype(np.float32)
            noise = self.rng.normal(0.0, 0.02, size=x.shape).astype(np.float32)
            x_aug = x * scale + noise
            if self.return_labels:
                return (x, x_aug), y
            return x, x_aug

        if self.return_labels:
            return x, y
        return x

def calculate_training_stats(frame, window_key):
    """Streaming mean/std over training windows only."""
    total_sum = 0.0
    total_sq_sum = 0.0
    total_count = 0

    for path, group in frame.groupby("output_path"):
        indices = group["window_index"].astype(int).to_numpy()
        with np.load(path) as data:
            windows = data[window_key][indices].astype(np.float64)

        total_sum += windows.sum(dtype=np.float64)
        total_sq_sum += np.square(windows).sum(dtype=np.float64)
        total_count += windows.size

    mean = total_sum / total_count
    variance = max(total_sq_sum / total_count - mean ** 2, 1e-12)
    std = float(np.sqrt(variance))
    return float(mean), std

print("Sequence class and fold-specific normalization helper defined.")
print("Raw window size: 20,000")
print("Order window size: 2,048")
print("\nCELL 3 COMPLETED")


Sequence class and fold-specific normalization helper defined.
Raw window size: 20,000
Order window size: 2,048

CELL 3 COMPLETED


In [19]:

# ==========================================================
# CELL 4 — MODEL ARCHITECTURE AND CONSISTENCY OBJECTIVE
# Objective: define the same compact CNN backbone for raw
# and order windows, plus a consistency-training wrapper.
# ==========================================================

from tensorflow.keras import layers, models

def build_cnn(input_length, model_name):
    inputs = layers.Input(shape=(input_length, 1), name="signal")

    x = layers.Conv1D(
        32, kernel_size=15, strides=4, padding="same",
        activation="relu", name="conv1"
    )(inputs)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.MaxPooling1D(pool_size=4, name="pool1")(x)

    x = layers.Conv1D(
        64, kernel_size=9, padding="same",
        activation="relu", name="conv2"
    )(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.MaxPooling1D(pool_size=4, name="pool2")(x)

    x = layers.Conv1D(
        128, kernel_size=7, padding="same",
        activation="relu", name="conv3"
    )(x)
    x = layers.BatchNormalization(name="bn3")(x)

    x = layers.GlobalAveragePooling1D(name="global_average_pooling")(x)
    x = layers.Dense(64, activation="relu", name="dense64")(x)
    x = layers.Dropout(0.4, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="fault_probability")(x)

    model = models.Model(inputs, outputs, name=model_name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    return model

class OrderConsistencyModel(tf.keras.Model):
    """
    Training objective:
      0.5 * (BCE(clean) + BCE(augmented))
      + 0.1 * MSE(clean_prediction, augmented_prediction)
    """
    def __init__(self, base_model, consistency_weight=0.1):
        super().__init__(name="order_consistency_model")
        self.base_model = base_model
        self.consistency_weight = float(consistency_weight)
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.accuracy_tracker = tf.keras.metrics.BinaryAccuracy(name="accuracy")
        self.auc_tracker = tf.keras.metrics.AUC(name="auc")
        self.bce = tf.keras.losses.BinaryCrossentropy()

    @property
    def metrics(self):
        return [self.loss_tracker, self.accuracy_tracker, self.auc_tracker]

    def call(self, inputs, training=False):
        return self.base_model(inputs, training=training)

    def train_step(self, data):
        (clean_x, augmented_x), y = data

        with tf.GradientTape() as tape:
            clean_p = self.base_model(clean_x, training=True)
            augmented_p = self.base_model(augmented_x, training=True)

            clean_loss = self.bce(y, clean_p)
            augmented_loss = self.bce(y, augmented_p)
            consistency_loss = tf.reduce_mean(tf.square(clean_p - augmented_p))

            total_loss = (
                0.5 * (clean_loss + augmented_loss)
                + self.consistency_weight * consistency_loss
            )

        gradients = tape.gradient(total_loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.base_model.trainable_variables))

        self.loss_tracker.update_state(total_loss)
        self.accuracy_tracker.update_state(y, clean_p)
        self.auc_tracker.update_state(y, clean_p)

        return {metric.name: metric.result() for metric in self.metrics}

    def test_step(self, data):
        x, y = data
        predictions = self.base_model(x, training=False)
        loss = self.bce(y, predictions)

        self.loss_tracker.update_state(loss)
        self.accuracy_tracker.update_state(y, predictions)
        self.auc_tracker.update_state(y, predictions)

        return {metric.name: metric.result() for metric in self.metrics}

class SaveBestBaseModel(tf.keras.callbacks.Callback):
    """Save the base CNN whenever validation AUC improves."""
    def __init__(self, filepath):
        super().__init__()
        self.filepath = filepath
        self.best = -np.inf

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        value = logs.get("val_auc")
        if value is not None and np.isfinite(value) and value > self.best:
            self.best = float(value)
            self.model.base_model.save(self.filepath)

def training_callbacks(checkpoint_path):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            verbose=1
        )
    ]

print("CNN architecture and consistency-training wrapper defined.")
print("\nCELL 4 COMPLETED")


CNN architecture and consistency-training wrapper defined.

CELL 4 COMPLETED


In [20]:

# ==========================================================
# CELL 5 — TRAIN RAW CNN ON ALL FOUR FOLDS
# Objective: train a fresh raw-vibration CNN for each
# held-out operating-condition fold.
# ==========================================================

EPOCHS = 40
RAW_BATCH_SIZE = 8

raw_histories = {}
raw_norm_stats = {}

for fold in range(1, 5):
    print("\n" + "=" * 75)
    print(f"TRAINING RAW CNN — FOLD {fold}")
    print("=" * 75)

    train_df = raw_manifest[(raw_manifest.fold == fold) & (raw_manifest.split == "train")].copy()
    val_df = raw_manifest[(raw_manifest.fold == fold) & (raw_manifest.split == "validation")].copy()

    mean, std = calculate_training_stats(train_df, RAW_WINDOW_KEY)
    raw_norm_stats[fold] = {"mean": mean, "std": std}

    print("Training windows:", len(train_df))
    print("Validation windows:", len(val_df))
    print("Training normalization mean:", mean)
    print("Training normalization std:", std)

    train_seq = WindowSequence(
        train_df, RAW_WINDOW_KEY, 20000, mean, std,
        batch_size=RAW_BATCH_SIZE, shuffle=True, seed=SEED + fold
    )
    val_seq = WindowSequence(
        val_df, RAW_WINDOW_KEY, 20000, mean, std,
        batch_size=RAW_BATCH_SIZE, shuffle=False, seed=SEED
    )

    tf.keras.backend.clear_session()
    model = build_cnn(20000, f"raw_cnn_fold{fold}")
    model.summary()

    checkpoint = os.path.join(RUN_MODELS, f"fold{fold}_raw_cnn_best.keras")

    history = model.fit(
        train_seq,
        validation_data=val_seq,
        epochs=EPOCHS,
        callbacks=training_callbacks(checkpoint),
        verbose=1
    )

    raw_histories[fold] = history.history

    with open(os.path.join(RUN_RESULTS, f"fold{fold}_raw_normalization.json"), "w") as f:
        json.dump(raw_norm_stats[fold], f, indent=2)

    pd.DataFrame(history.history).to_csv(
        os.path.join(RUN_RESULTS, f"fold{fold}_raw_training_history.csv"),
        index=False
    )

    print("Saved:", checkpoint)

print("\nALL FOUR RAW CNN FOLDS TRAINED")



TRAINING RAW CNN — FOLD 1
Training windows: 6628
Validation windows: 1656
Training normalization mean: 0.005046434261576848
Training normalization std: 0.3247729544682251


Model: "raw_cnn_fold1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ signal (InputLayer)             │ (None, 20000, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 5000, 32)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 5000, 32)       │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling1D)            │ (None, 1250, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 1250, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 1250, 64)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling1D)            │ (None, 312, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv1D)                  │ (None, 312, 128)       │        57,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 312, 128)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense64 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fault_probability (Dense)       │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,697 (334.75 KB)

 Trainable params: 85,249 (333.00 KB)

 Non-trainable params: 448 (1.75 KB)

... (15 outputs left out)

 Total params: 85,697 (334.75 KB)

 Trainable params: 85,249 (333.00 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/40
709/709 ━━━━━━━━━━━━━━━━━━━━ 0s 342ms/step - accuracy: 0.9605 - auc: 0.9886 - loss: 0.1116
Epoch 1: val_auc improved from None to 1.00000, saving model to /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/models/fold4_raw_cnn_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/models/fold4_raw_cnn_best.keras
709/709 ━━━━━━━━━━━━━━━━━━━━ 284s 396ms/step - accuracy: 0.9825 - auc: 0.9967 - loss: 0.0590 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.0043 - learning_rate: 0.0010
Epoch 2/40
709/709 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.9948 - auc: 0.9998 - loss: 0.0152
Epoch 2: val_auc did not improve from 1.00000
709/709 ━━━━━━━━━━━━━━━━━━━━ 259s 365ms/step - accuracy: 0.9931 - auc: 0.9996 - loss: 0.0196 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 8.8040e-05 - learning_rate: 0.0010
Epoch 3/40
709/709 ━━━━━━━━━━━━━━━━━━━━ 0s 328ms/step - accuracy: 0.9952 - auc: 0.9998 - loss

In [21]:

# ==========================================================
# CELL 6 — TRAIN ORDER CNN ON ALL FOUR FOLDS
# Objective: train the same CNN architecture on order-domain
# windows of 2,048 samples (8 revolutions × 256 samples/rev).
# ==========================================================

ORDER_BATCH_SIZE = 32

order_histories = {}
order_norm_stats = {}

for fold in range(1, 5):
    print("\n" + "=" * 75)
    print(f"TRAINING ORDER CNN — FOLD {fold}")
    print("=" * 75)

    train_df = order_manifest[
        (order_manifest.fold == fold) & (order_manifest.split == "train")
    ].copy()
    val_df = order_manifest[
        (order_manifest.fold == fold) & (order_manifest.split == "validation")
    ].copy()

    mean, std = calculate_training_stats(train_df, ORDER_WINDOW_KEY)
    order_norm_stats[fold] = {"mean": mean, "std": std}

    print("Training windows:", len(train_df))
    print("Validation windows:", len(val_df))
    print("Training normalization mean:", mean)
    print("Training normalization std:", std)

    train_seq = WindowSequence(
        train_df, ORDER_WINDOW_KEY, 2048, mean, std,
        batch_size=ORDER_BATCH_SIZE, shuffle=True, seed=SEED + fold
    )
    val_seq = WindowSequence(
        val_df, ORDER_WINDOW_KEY, 2048, mean, std,
        batch_size=ORDER_BATCH_SIZE, shuffle=False, seed=SEED
    )

    tf.keras.backend.clear_session()
    model = build_cnn(2048, f"order_cnn_fold{fold}")
    model.summary()

    checkpoint = os.path.join(RUN_MODELS, f"fold{fold}_order_cnn_best.keras")

    history = model.fit(
        train_seq,
        validation_data=val_seq,
        epochs=EPOCHS,
        callbacks=training_callbacks(checkpoint),
        verbose=1
    )

    order_histories[fold] = history.history

    with open(os.path.join(RUN_RESULTS, f"fold{fold}_order_normalization.json"), "w") as f:
        json.dump(order_norm_stats[fold], f, indent=2)

    pd.DataFrame(history.history).to_csv(
        os.path.join(RUN_RESULTS, f"fold{fold}_order_training_history.csv"),
        index=False
    )

    print("Saved:", checkpoint)

print("\nALL FOUR ORDER CNN FOLDS TRAINED")



TRAINING ORDER CNN — FOLD 1
Training windows: 6628
Validation windows: 1656
Training normalization mean: 0.004973001078062809
Training normalization std: 0.32441127962602123


Model: "order_cnn_fold1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ signal (InputLayer)             │ (None, 2048, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 512, 32)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 512, 32)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling1D)            │ (None, 128, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 128, 64)        │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 128, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling1D)            │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv1D)                  │ (None, 32, 128)        │        57,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 32, 128)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense64 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fault_probability (Dense)       │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,697 (334.75 KB)

 Trainable params: 85,249 (333.00 KB)

 Non-trainable params: 448 (1.75 KB)

... (15 outputs left out)

 Total params: 85,697 (334.75 KB)

 Trainable params: 85,249 (333.00 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/40
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step - accuracy: 0.9628 - auc: 0.9895 - loss: 0.1089
Epoch 1: val_auc improved from None to 0.75882, saving model to /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/models/fold4_order_cnn_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/models/fold4_order_cnn_best.keras
178/178 ━━━━━━━━━━━━━━━━━━━━ 53s 276ms/step - accuracy: 0.9850 - auc: 0.9986 - loss: 0.0481 - val_accuracy: 0.4996 - val_auc: 0.7588 - val_loss: 2.6566 - learning_rate: 0.0010
Epoch 2/40
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step - accuracy: 0.9964 - auc: 0.9999 - loss: 0.0112
Epoch 2: val_auc improved from 0.75882 to 0.96869, saving model to /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/models/fold4_order_cnn_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/models/fold4_order_cnn_best.keras
178/178 ━━━

In [22]:

# ==========================================================
# CELL 7 — TRAIN ORDER CNN WITH CONSISTENCY OBJECTIVE
# Objective: train an order-domain CNN using paired clean and
# augmented windows, with an additional prediction-consistency
# penalty. This cell trains all four folds for a fair comparison.
# ==========================================================

CONSISTENCY_BATCH_SIZE = 16
CONSISTENCY_WEIGHT = 0.1

consistency_histories = {}
consistency_norm_stats = {}

for fold in range(1, 5):
    print("\n" + "=" * 75)
    print(f"TRAINING ORDER CNN + CONSISTENCY — FOLD {fold}")
    print("=" * 75)

    train_df = order_manifest[
        (order_manifest.fold == fold) & (order_manifest.split == "train")
    ].copy()
    val_df = order_manifest[
        (order_manifest.fold == fold) & (order_manifest.split == "validation")
    ].copy()

    mean, std = calculate_training_stats(train_df, ORDER_WINDOW_KEY)
    consistency_norm_stats[fold] = {"mean": mean, "std": std}

    train_seq = WindowSequence(
        train_df, ORDER_WINDOW_KEY, 2048, mean, std,
        batch_size=CONSISTENCY_BATCH_SIZE,
        shuffle=True,
        augment=True,
        return_labels=True,
        seed=SEED + 100 + fold
    )
    val_seq = WindowSequence(
        val_df, ORDER_WINDOW_KEY, 2048, mean, std,
        batch_size=CONSISTENCY_BATCH_SIZE,
        shuffle=False,
        augment=False,
        return_labels=True,
        seed=SEED
    )

    tf.keras.backend.clear_session()

    base_model = build_cnn(2048, f"order_consistency_base_fold{fold}")
    consistency_model = OrderConsistencyModel(
        base_model,
        consistency_weight=CONSISTENCY_WEIGHT
    )
    consistency_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)
    )

    checkpoint = os.path.join(
        RUN_MODELS, f"fold{fold}_order_consistency_best.keras"
    )

    callbacks = [
        SaveBestBaseModel(checkpoint),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=8,
            restore_best_weights=False,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            verbose=1
        )
    ]

    history = consistency_model.fit(
        train_seq,
        validation_data=val_seq,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    consistency_histories[fold] = history.history

    with open(
        os.path.join(RUN_RESULTS, f"fold{fold}_consistency_normalization.json"), "w"
    ) as f:
        json.dump(consistency_norm_stats[fold], f, indent=2)

    pd.DataFrame(history.history).to_csv(
        os.path.join(RUN_RESULTS, f"fold{fold}_consistency_training_history.csv"),
        index=False
    )

    print("Saved best base CNN:", checkpoint)

print("\nALL FOUR CONSISTENCY FOLDS TRAINED")



TRAINING ORDER CNN + CONSISTENCY — FOLD 1
Epoch 1/40
415/415 ━━━━━━━━━━━━━━━━━━━━ 91s 208ms/step - accuracy: 0.9920 - auc: 0.9990 - loss: 0.0299 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 2.4591e-04 - learning_rate: 0.0010
Epoch 2/40
415/415 ━━━━━━━━━━━━━━━━━━━━ 84s 203ms/step - accuracy: 0.9983 - auc: 1.0000 - loss: 0.0059 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 1.2312e-04 - learning_rate: 0.0010
Epoch 3/40
415/415 ━━━━━━━━━━━━━━━━━━━━ 76s 182ms/step - accuracy: 0.9976 - auc: 0.9998 - loss: 0.0084 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 2.0031e-05 - learning_rate: 0.0010
Epoch 4/40
415/415 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 0.9995 - auc: 0.9999 - loss: 0.0024
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
415/415 ━━━━━━━━━━━━━━━━━━━━ 81s 181ms/step - accuracy: 0.9992 - auc: 0.9998 - loss: 0.0031 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 1.0809e-05 - learning_rate: 0.0010
Epoch 5/40
415/415 ━━━━━━━

In [23]:

# ==========================================================
# CELL 8 — RECORDING-LEVEL PREDICTIONS FOR ALL MODELS
# Objective: predict every held-out test window, then average
# window probabilities to obtain one prediction per recording.
# ==========================================================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, roc_auc_score, confusion_matrix
)

def predict_recordings(model, test_df, window_key, window_size, mean, std, batch_size=16):
    prediction_seq = WindowSequence(
        test_df, window_key, window_size, mean, std,
        batch_size=batch_size,
        shuffle=False,
        augment=False,
        return_labels=False
    )

    window_probabilities = model.predict(prediction_seq, verbose=1).reshape(-1)

    if len(window_probabilities) != len(test_df):
        raise ValueError("Prediction count does not match test-window count.")

    window_output = test_df[
        ["filename", "bearing_id", "label", "fold", "held_out_condition", "window_index"]
    ].copy()
    window_output["window_probability"] = window_probabilities

    recording_output = (
        window_output.groupby(
            ["filename", "bearing_id", "label", "fold", "held_out_condition"],
            as_index=False
        )
        .agg(
            mean_probability=("window_probability", "mean"),
            number_of_windows=("window_probability", "size")
        )
    )

    recording_output["predicted_label"] = (
        recording_output["mean_probability"] >= 0.5
    ).astype(int)

    return recording_output

def calculate_metrics(predictions):
    y_true = predictions["label"].astype(int).to_numpy()
    y_pred = predictions["predicted_label"].astype(int).to_numpy()
    y_prob = predictions["mean_probability"].astype(float).to_numpy()

    return {
        "test_recordings": len(predictions),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_damaged": precision_score(y_true, y_pred, zero_division=0),
        "damaged_recall": recall_score(y_true, y_pred, zero_division=0),
        "damaged_f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "healthy_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    }

all_predictions = []
all_fold_metrics = []

for fold in range(1, 5):
    print("\n" + "=" * 75)
    print(f"GENERATING TEST PREDICTIONS — FOLD {fold}")
    print("=" * 75)

    raw_test = raw_manifest[
        (raw_manifest.fold == fold) & (raw_manifest.split == "test")
    ].copy()
    order_test = order_manifest[
        (order_manifest.fold == fold) & (order_manifest.split == "test")
    ].copy()

    # RAW CNN
    raw_model = tf.keras.models.load_model(
        os.path.join(RUN_MODELS, f"fold{fold}_raw_cnn_best.keras")
    )
    raw_pred = predict_recordings(
        raw_model, raw_test, RAW_WINDOW_KEY, 20000,
        raw_norm_stats[fold]["mean"], raw_norm_stats[fold]["std"],
        batch_size=RAW_BATCH_SIZE
    )
    raw_pred["model"] = "Raw CNN"
    all_predictions.append(raw_pred)
    all_fold_metrics.append({"fold": fold, "model": "Raw CNN", **calculate_metrics(raw_pred)})

    del raw_model
    tf.keras.backend.clear_session()

    # ORDER CNN
    order_model = tf.keras.models.load_model(
        os.path.join(RUN_MODELS, f"fold{fold}_order_cnn_best.keras")
    )
    order_pred = predict_recordings(
        order_model, order_test, ORDER_WINDOW_KEY, 2048,
        order_norm_stats[fold]["mean"], order_norm_stats[fold]["std"],
        batch_size=ORDER_BATCH_SIZE
    )
    order_pred["model"] = "Order CNN"
    all_predictions.append(order_pred)
    all_fold_metrics.append({"fold": fold, "model": "Order CNN", **calculate_metrics(order_pred)})

    del order_model
    tf.keras.backend.clear_session()

    # ORDER CNN + CONSISTENCY
    consistency_model = tf.keras.models.load_model(
        os.path.join(RUN_MODELS, f"fold{fold}_order_consistency_best.keras")
    )
    consistency_pred = predict_recordings(
        consistency_model, order_test, ORDER_WINDOW_KEY, 2048,
        consistency_norm_stats[fold]["mean"], consistency_norm_stats[fold]["std"],
        batch_size=ORDER_BATCH_SIZE
    )
    consistency_pred["model"] = "Order CNN + Consistency"
    all_predictions.append(consistency_pred)
    all_fold_metrics.append({
        "fold": fold,
        "model": "Order CNN + Consistency",
        **calculate_metrics(consistency_pred)
    })

    del consistency_model
    tf.keras.backend.clear_session()

predictions_df = pd.concat(all_predictions, ignore_index=True)
fold_metrics_df = pd.DataFrame(all_fold_metrics)

predictions_df.to_csv(
    os.path.join(RUN_PREDICTIONS, "all_recording_level_predictions.csv"),
    index=False
)
fold_metrics_df.to_csv(
    os.path.join(RUN_RESULTS, "all_fold_metrics.csv"),
    index=False
)

print("\nPREDICTION COUNTS")
print(predictions_df.groupby(["model", "fold"]).size())

print("\nFOLD METRICS")
display(fold_metrics_df.round(4))

print("\nCELL 8 COMPLETED")



GENERATING TEST PREDICTIONS — FOLD 1
196/196 ━━━━━━━━━━━━━━━━━━━━ 32s 162ms/step
49/49 ━━━━━━━━━━━━━━━━━━━━ 7s 131ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 34 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


49/49 ━━━━━━━━━━━━━━━━━━━━ 8s 152ms/step

GENERATING TEST PREDICTIONS — FOLD 2
345/345 ━━━━━━━━━━━━━━━━━━━━ 83s 238ms/step
87/87 ━━━━━━━━━━━━━━━━━━━━ 16s 181ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 34 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


87/87 ━━━━━━━━━━━━━━━━━━━━ 16s 183ms/step

GENERATING TEST PREDICTIONS — FOLD 3
346/346 ━━━━━━━━━━━━━━━━━━━━ 80s 230ms/step
87/87 ━━━━━━━━━━━━━━━━━━━━ 16s 185ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 34 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


87/87 ━━━━━━━━━━━━━━━━━━━━ 16s 184ms/step

GENERATING TEST PREDICTIONS — FOLD 4
346/346 ━━━━━━━━━━━━━━━━━━━━ 87s 248ms/step
87/87 ━━━━━━━━━━━━━━━━━━━━ 16s 187ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 34 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


87/87 ━━━━━━━━━━━━━━━━━━━━ 18s 196ms/step

PREDICTION COUNTS
model                    fold
Order CNN                1       120
                         2       120
                         3       120
                         4       120
Order CNN + Consistency  1       120
                         2       120
                         3       120
                         4       120
Raw CNN                  1       120
                         2       120
                         3       120
                         4       120
dtype: int64

FOLD METRICS


,fold,model,test_recordings,accuracy,precision_damaged,damaged_recall,damaged_f1,macro_f1,balanced_accuracy,auc,healthy_recall
0,1,Raw CNN,120,0.8333,1.0,0.6667,0.8000,0.8286,0.8333,1.0000,1.0
1,1,Order CNN,120,0.5667,1.0,0.1333,0.2353,0.4665,0.5667,0.7881,1.0
2,1,Order CNN + Consistency,120,0.8333,1.0,0.6667,0.8000,0.8286,0.8333,0.9703,1.0
3,2,Raw CNN,120,1.0000,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0
4,2,Order CNN,120,1.0000,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0
5,2,Order CNN + Consistency,120,1.0000,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0
6,3,Raw CNN,120,1.0000,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0
7,3,Order CNN,120,1.0000,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0
8,3,Order CNN + Consistency,120,1.0000,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0
9,4,Raw CNN,120,1.0000,1.0,1.0000,1.0000,1.0000,1.0000,1.0000,1.0



CELL 8 COMPLETED


In [24]:

# ==========================================================
# CELL 9 — FOUR-FOLD SUMMARY, CONFUSION MATRICES, AND EXPORT
# Objective: calculate mean and standard deviation across
# folds, produce confusion matrices, and save final tables.
# ==========================================================

metric_columns = [
    "accuracy", "precision_damaged", "damaged_recall",
    "damaged_f1", "macro_f1", "balanced_accuracy",
    "auc", "healthy_recall"
]

mean_summary = (
    fold_metrics_df.groupby("model")[metric_columns]
    .agg(["mean", "std"])
    .reset_index()
)

mean_summary.columns = [
    "_".join([str(x) for x in col if str(x) != ""]).strip("_")
    if isinstance(col, tuple) else col
    for col in mean_summary.columns
]

mean_summary.to_csv(
    os.path.join(RUN_RESULTS, "four_fold_mean_and_std_metrics.csv"),
    index=False
)

confusion_rows = []

for (model_name, fold), group in predictions_df.groupby(["model", "fold"]):
    cm = confusion_matrix(
        group["label"].astype(int),
        group["predicted_label"].astype(int),
        labels=[0, 1]
    )

    confusion_rows.append({
        "model": model_name,
        "fold": fold,
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1])
    })

confusion_df = pd.DataFrame(confusion_rows)
confusion_df.to_csv(
    os.path.join(RUN_RESULTS, "four_fold_confusion_matrices.csv"),
    index=False
)

print("=" * 75)
print("FOUR-FOLD MEAN AND STANDARD DEVIATION")
print("=" * 75)
display(mean_summary.round(4))

print("\nCONFUSION MATRICES")
display(confusion_df)

print("\nSAVED OUTPUTS")
for folder in [RUN_MODELS, RUN_PREDICTIONS, RUN_RESULTS]:
    print("\n", folder)
    for filename in sorted(os.listdir(folder)):
        print(" -", filename)

print("\nFRESH TRAINING AND PREDICTION RUN COMPLETED")


FOUR-FOLD MEAN AND STANDARD DEVIATION


,model,accuracy_mean,accuracy_std,precision_damaged_mean,precision_damaged_std,damaged_recall_mean,damaged_recall_std,damaged_f1_mean,damaged_f1_std,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,balanced_accuracy_std,auc_mean,auc_std,healthy_recall_mean,healthy_recall_std
0,Order CNN,0.8917,0.2167,1.0,0.0,0.7833,0.4333,0.8088,0.3824,0.8666,0.2668,0.8917,0.2167,0.9470,0.1060,1.0,0.0
1,Order CNN + Consistency,0.9583,0.0833,1.0,0.0,0.9167,0.1667,0.9500,0.1000,0.9571,0.0857,0.9583,0.0833,0.9926,0.0149,1.0,0.0
2,Raw CNN,0.9583,0.0833,1.0,0.0,0.9167,0.1667,0.9500,0.1000,0.9571,0.0857,0.9583,0.0833,1.0000,0.0000,1.0,0.0



CONFUSION MATRICES


,model,fold,TN,FP,FN,TP
0,Order CNN,1,60,0,52,8
1,Order CNN,2,60,0,0,60
2,Order CNN,3,60,0,0,60
3,Order CNN,4,60,0,0,60
4,Order CNN + Consistency,1,60,0,20,40
5,Order CNN + Consistency,2,60,0,0,60
6,Order CNN + Consistency,3,60,0,0,60
7,Order CNN + Consistency,4,60,0,0,60
8,Raw CNN,1,60,0,20,40
9,Raw CNN,2,60,0,0,60



SAVED OUTPUTS

 /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/models
 - fold1_order_cnn_best.keras
 - fold1_order_consistency_best.keras
 - fold1_raw_cnn_best.keras
 - fold2_order_cnn_best.keras
 - fold2_order_consistency_best.keras
 - fold2_raw_cnn_best.keras
 - fold3_order_cnn_best.keras
 - fold3_order_consistency_best.keras
 - fold3_raw_cnn_best.keras
 - fold4_order_cnn_best.keras
 - fold4_order_consistency_best.keras
 - fold4_raw_cnn_best.keras

 /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/predictions
 - all_recording_level_predictions.csv

 /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results
 - all_fold_metrics.csv
 - fold1_consistency_normalization.json
 - fold1_consistency_training_history.csv
 - fold1_order_normalization.json
 - fold1_order_training_history.csv
 - fold1_raw_normalization.json
 - fold1_raw_training_history.csv
 - fold2_consistency_normalization.json
 - fold2_consistency_training_history.csv
 

In [25]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 1: INSPECT SAVED EXPERIMENT FILES
#
# Objective:
# Load the saved manifests, predictions, and metrics.
# Inspect their structures without modifying any files.
#
# Expected outcome:
# Five CSV files load successfully, with their shapes,
# columns, and sample records displayed.
# ============================================================

import os
import pandas as pd

BASE = (
    "/content/drive/MyDrive/BearingFaultResearch/"
    "fresh_train_predict_run"
)

FILES = {
    "split_manifest": (
        "results/recovered_recording_split_manifest.csv"
    ),
    "raw_manifest": (
        "results/fresh_raw_window_manifest.csv"
    ),
    "order_manifest": (
        "results/fresh_order_window_manifest.csv"
    ),
    "predictions": (
        "predictions/all_recording_level_predictions.csv"
    ),
    "fold_metrics": (
        "results/all_fold_metrics.csv"
    )
}

audit_data = {}

print("=" * 75)
print("PROJECT 19 — STAGE 1: SAVED FILE INSPECTION")
print("=" * 75)

for name, relative_path in FILES.items():

    path = os.path.join(BASE, relative_path)

    print(f"\n{'=' * 75}")
    print(f"FILE: {name}")
    print(f"PATH: {path}")

    if not os.path.exists(path):
        print("STATUS: FILE NOT FOUND")
        continue

    df = pd.read_csv(path)
    audit_data[name] = df

    print("STATUS: LOADED")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 3 records:")
    display(df.head(3))

print("\n" + "=" * 75)
print("STAGE 1 INSPECTION COMPLETED")
print("Successfully loaded:", list(audit_data.keys()))
print("=" * 75)

PROJECT 19 — STAGE 1: SAVED FILE INSPECTION

FILE: split_manifest
PATH: /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/recovered_recording_split_manifest.csv
STATUS: LOADED
Shape: (1920, 6)

Columns:
['filename', 'bearing_id', 'label', 'fold', 'split', 'held_out_condition']

First 3 records:


,filename,bearing_id,label,fold,split,held_out_condition
0,N09_M07_F10_K001_1.mat,K001,0,1,test,N09_M07_F10
1,N09_M07_F10_K001_10.mat,K001,0,1,test,N09_M07_F10
2,N09_M07_F10_K001_11.mat,K001,0,1,test,N09_M07_F10



FILE: raw_manifest
PATH: /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fresh_raw_window_manifest.csv
STATUS: LOADED
Shape: (39396, 8)

Columns:
['filename', 'bearing_id', 'label', 'fold', 'split', 'held_out_condition', 'window_index', 'output_path']

First 3 records:


,filename,bearing_id,label,fold,split,held_out_condition,window_index,output_path
0,N15_M07_F10_KI04_18.mat,KI04,1,1,train,N09_M07_F10,0,/content/drive/MyDrive/BearingFaultResearch/da...
1,N15_M07_F10_KI04_18.mat,KI04,1,1,train,N09_M07_F10,1,/content/drive/MyDrive/BearingFaultResearch/da...
2,N15_M07_F10_KI04_18.mat,KI04,1,1,train,N09_M07_F10,2,/content/drive/MyDrive/BearingFaultResearch/da...



FILE: order_manifest
PATH: /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fresh_order_window_manifest.csv
STATUS: LOADED
Shape: (39396, 8)

Columns:
['filename', 'bearing_id', 'label', 'fold', 'split', 'held_out_condition', 'window_index', 'output_path']

First 3 records:


,filename,bearing_id,label,fold,split,held_out_condition,window_index,output_path
0,N15_M07_F10_KI04_18.mat,KI04,1,1,train,N09_M07_F10,0,/content/drive/MyDrive/BearingFaultResearch/da...
1,N15_M07_F10_KI04_18.mat,KI04,1,1,train,N09_M07_F10,1,/content/drive/MyDrive/BearingFaultResearch/da...
2,N15_M07_F10_KI04_18.mat,KI04,1,1,train,N09_M07_F10,2,/content/drive/MyDrive/BearingFaultResearch/da...



FILE: predictions
PATH: /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/predictions/all_recording_level_predictions.csv
STATUS: LOADED
Shape: (1440, 9)

Columns:
['filename', 'bearing_id', 'label', 'fold', 'held_out_condition', 'mean_probability', 'number_of_windows', 'predicted_label', 'model']

First 3 records:


,filename,bearing_id,label,fold,held_out_condition,mean_probability,number_of_windows,predicted_label,model
0,N09_M07_F10_K001_1.mat,K001,0,1,N09_M07_F10,0.000008,13,0,Raw CNN
1,N09_M07_F10_K001_10.mat,K001,0,1,N09_M07_F10,0.000008,13,0,Raw CNN
2,N09_M07_F10_K001_11.mat,K001,0,1,N09_M07_F10,0.000009,13,0,Raw CNN



FILE: fold_metrics
PATH: /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/all_fold_metrics.csv
STATUS: LOADED
Shape: (12, 11)

Columns:
['fold', 'model', 'test_recordings', 'accuracy', 'precision_damaged', 'damaged_recall', 'damaged_f1', 'macro_f1', 'balanced_accuracy', 'auc', 'healthy_recall']

First 3 records:


,fold,model,test_recordings,accuracy,precision_damaged,damaged_recall,damaged_f1,macro_f1,balanced_accuracy,auc,healthy_recall
0,1,Raw CNN,120,0.833333,1.0,0.666667,0.800000,0.828571,0.833333,1.000000,1.0
1,1,Order CNN,120,0.566667,1.0,0.133333,0.235294,0.466484,0.566667,0.788056,1.0
2,1,Order CNN + Consistency,120,0.833333,1.0,0.666667,0.800000,0.828571,0.833333,0.970278,1.0



STAGE 1 INSPECTION COMPLETED
Successfully loaded: ['split_manifest', 'raw_manifest', 'order_manifest', 'predictions', 'fold_metrics']


In [26]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 2: RECORDING-LEVEL SPLIT INTEGRITY
#
# Objective:
# Verify that each recording belongs to exactly one split
# within each fold, and confirm expected recording counts.
#
# Expected outcome:
# 288 train, 72 validation, 120 test recordings per fold.
# No recording should appear in multiple splits.
# ============================================================

import pandas as pd

split_df = audit_data["split_manifest"].copy()

print("=" * 75)
print("PROJECT 19 — STAGE 2: RECORDING-LEVEL SPLIT AUDIT")
print("=" * 75)

# Check duplicate recording assignments within each fold
duplicate_assignments = (
    split_df.groupby(["fold", "filename"])["split"]
    .nunique()
)

overlapping_recordings = duplicate_assignments[
    duplicate_assignments > 1
]

print("\n1. RECORDING OVERLAP CHECK")
print("Recordings assigned to multiple splits:", len(overlapping_recordings))

# Check expected counts
print("\n2. RECORDING COUNTS PER FOLD")

split_counts = (
    split_df.groupby(["fold", "split"])["filename"]
    .nunique()
    .unstack(fill_value=0)
)

display(split_counts)

# Check total unique recordings in each fold
print("\n3. UNIQUE RECORDINGS PER FOLD")

fold_totals = split_df.groupby("fold")["filename"].nunique()
display(fold_totals)

# Check repeated rows in manifest
print("\n4. DUPLICATE MANIFEST ROWS")
print("Exact duplicate rows:", split_df.duplicated().sum())

# Check whether filename has conflicting labels
print("\n5. RECORDING LABEL CONSISTENCY")

label_counts = (
    split_df.groupby("filename")["label"]
    .nunique()
)

conflicting_labels = label_counts[label_counts > 1]

print("Recordings with conflicting labels:", len(conflicting_labels))

# Overall status
print("\n" + "=" * 75)
print("STAGE 2 SUMMARY")
print("=" * 75)

if len(overlapping_recordings) == 0:
    print("PASS: No recording appears in multiple splits within a fold.")
else:
    print("WARNING: Recording overlap detected.")

if len(conflicting_labels) == 0:
    print("PASS: No recording has conflicting labels.")
else:
    print("WARNING: Conflicting recording labels detected.")

print("\nSTAGE 2 COMPLETED — REVIEW COUNTS BEFORE PROCEEDING")

PROJECT 19 — STAGE 2: RECORDING-LEVEL SPLIT AUDIT

1. RECORDING OVERLAP CHECK
Recordings assigned to multiple splits: 0

2. RECORDING COUNTS PER FOLD


split,test,train,validation
fold,,,
1,120,288,72
2,120,288,72
3,120,288,72
4,120,288,72



3. UNIQUE RECORDINGS PER FOLD


,filename
fold,
1,480
2,480
3,480
4,480



4. DUPLICATE MANIFEST ROWS
Exact duplicate rows: 0

5. RECORDING LABEL CONSISTENCY
Recordings with conflicting labels: 0

STAGE 2 SUMMARY
PASS: No recording appears in multiple splits within a fold.
PASS: No recording has conflicting labels.

STAGE 2 COMPLETED — REVIEW COUNTS BEFORE PROCEEDING


In [27]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 3: RAW AND ORDER WINDOW-MANIFEST INTEGRITY
#
# Objective:
# Confirm that raw and order-domain windows represent the
# same recordings, labels, folds, splits, and window indices.
# Also check for duplicate window records and output paths.
# ============================================================

import pandas as pd

raw_df = audit_data["raw_manifest"].copy()
order_df = audit_data["order_manifest"].copy()

print("=" * 75)
print("PROJECT 19 — STAGE 3: WINDOW-MANIFEST INTEGRITY")
print("=" * 75)

# Columns that should identify the same window in both datasets
key_columns = [
    "filename",
    "bearing_id",
    "label",
    "fold",
    "split",
    "held_out_condition",
    "window_index"
]

# 1. Check that required columns exist
missing_raw = [c for c in key_columns + ["output_path"] if c not in raw_df.columns]
missing_order = [c for c in key_columns + ["output_path"] if c not in order_df.columns]

print("\n1. REQUIRED COLUMN CHECK")
print("Missing raw-manifest columns:", missing_raw)
print("Missing order-manifest columns:", missing_order)

if missing_raw or missing_order:
    raise ValueError("Required columns are missing. Stop and share this output.")

# 2. Check duplicate window identities
print("\n2. DUPLICATE WINDOW-IDENTITY CHECK")

raw_duplicate_keys = raw_df.duplicated(subset=key_columns).sum()
order_duplicate_keys = order_df.duplicated(subset=key_columns).sum()

print("Duplicate raw window identities:", raw_duplicate_keys)
print("Duplicate order window identities:", order_duplicate_keys)

# 3. Compare raw and order window identities
print("\n3. RAW-TO-ORDER WINDOW MATCHING")

raw_keys = raw_df[key_columns].drop_duplicates()
order_keys = order_df[key_columns].drop_duplicates()

comparison = raw_keys.merge(
    order_keys,
    on=key_columns,
    how="outer",
    indicator=True
)

raw_only = (comparison["_merge"] == "left_only").sum()
order_only = (comparison["_merge"] == "right_only").sum()
matched = (comparison["_merge"] == "both").sum()

print("Matched window identities:", matched)
print("Windows found only in raw manifest:", raw_only)
print("Windows found only in order manifest:", order_only)

# 4. Compare number of windows per recording and fold
print("\n4. WINDOWS PER RECORDING COMPARISON")

group_columns = ["fold", "filename", "split"]

raw_counts = (
    raw_df.groupby(group_columns)["window_index"]
    .nunique()
    .rename("raw_windows")
)

order_counts = (
    order_df.groupby(group_columns)["window_index"]
    .nunique()
    .rename("order_windows")
)

window_count_comparison = pd.concat(
    [raw_counts, order_counts],
    axis=1
)

window_count_comparison["counts_match"] = (
    window_count_comparison["raw_windows"]
    == window_count_comparison["order_windows"]
)

count_mismatches = (
    window_count_comparison["counts_match"] == False
).sum()

print("Recording/fold groups checked:", len(window_count_comparison))
print("Groups with different window counts:", count_mismatches)

# 5. Check output-path uniqueness
print("\n5. OUTPUT-PATH DUPLICATE CHECK")

raw_duplicate_paths = raw_df["output_path"].duplicated().sum()
order_duplicate_paths = order_df["output_path"].duplicated().sum()

print("Duplicate raw output paths:", raw_duplicate_paths)
print("Duplicate order output paths:", order_duplicate_paths)

# 6. Check whether each recording has consistent labels
print("\n6. LABEL CONSISTENCY WITHIN EACH MANIFEST")

raw_label_conflicts = (
    raw_df.groupby("filename")["label"].nunique() > 1
).sum()

order_label_conflicts = (
    order_df.groupby("filename")["label"].nunique() > 1
).sum()

print("Raw-manifest label conflicts:", raw_label_conflicts)
print("Order-manifest label conflicts:", order_label_conflicts)

# 7. Summary
print("\n" + "=" * 75)
print("STAGE 3 SUMMARY")
print("=" * 75)

checks = {
    "No duplicate raw window identities": raw_duplicate_keys == 0,
    "No duplicate order window identities": order_duplicate_keys == 0,
    "All raw windows match order windows": raw_only == 0 and order_only == 0,
    "Window counts match per recording/fold": count_mismatches == 0,
    "Raw output paths are unique": raw_duplicate_paths == 0,
    "Order output paths are unique": order_duplicate_paths == 0,
    "No raw label conflicts": raw_label_conflicts == 0,
    "No order label conflicts": order_label_conflicts == 0
}

for check, passed in checks.items():
    print(("PASS: " if passed else "WARNING: ") + check)

print("\nSTAGE 3 COMPLETED — SEND ME THE FULL OUTPUT")

PROJECT 19 — STAGE 3: WINDOW-MANIFEST INTEGRITY

1. REQUIRED COLUMN CHECK
Missing raw-manifest columns: []
Missing order-manifest columns: []

2. DUPLICATE WINDOW-IDENTITY CHECK
Duplicate raw window identities: 0
Duplicate order window identities: 0

3. RAW-TO-ORDER WINDOW MATCHING
Matched window identities: 39396
Windows found only in raw manifest: 0
Windows found only in order manifest: 0

4. WINDOWS PER RECORDING COMPARISON
Recording/fold groups checked: 1920
Groups with different window counts: 0

5. OUTPUT-PATH DUPLICATE CHECK
Duplicate raw output paths: 38916
Duplicate order output paths: 38916

6. LABEL CONSISTENCY WITHIN EACH MANIFEST
Raw-manifest label conflicts: 0
Order-manifest label conflicts: 0

STAGE 3 SUMMARY
PASS: No duplicate raw window identities
PASS: No duplicate order window identities
PASS: All raw windows match order windows
PASS: Window counts match per recording/fold
PASS: No raw label conflicts
PASS: No order label conflicts

STAGE 3 COMPLETED — SEND ME THE FU

In [29]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 3B: OUTPUT-PATH COLLISION INVESTIGATION
#
# Objective:
# Check whether repeated output paths refer to the same
# recording/window across folds or to different windows.
# ============================================================

import pandas as pd

print("=" * 75)
print("PROJECT 19 — STAGE 3B: OUTPUT-PATH INVESTIGATION")
print("=" * 75)

for name, df in [
    ("RAW", audit_data["raw_manifest"]),
    ("ORDER", audit_data["order_manifest"])
]:

    print(f"\n{'=' * 25} {name} MANIFEST {'=' * 25}")

    # Count distinct recording/window combinations per path
    path_mapping = (
        df.groupby("output_path")
        .apply(
            lambda group: pd.Series({
                "recordings": group["filename"].nunique(),
                "recording_window_pairs": group[
                    ["filename", "window_index"]
                ].drop_duplicates().shape[0],
                "folds": group["fold"].nunique(),
                "rows": len(group)
            }),
            include_groups=False
        )
    )

    # A collision means one path is associated with
    # more than one distinct recording/window pair.
    collisions = path_mapping[
        path_mapping["recording_window_pairs"] > 1
    ]

    # Paths reused across folds but representing the same
    # recording/window are expected in this experiment.
    reused_across_folds = path_mapping[
        (path_mapping["folds"] > 1) &
        (path_mapping["recording_window_pairs"] == 1)
    ]

    print("Unique output paths:", len(path_mapping))
    print("Paths reused across folds, same window:", len(reused_across_folds))
    print("Paths associated with different windows:", len(collisions))

    if len(collisions) > 0:
        print("\nExamples of path collisions:")
        display(collisions.head(10))

    print("\nSample repeated paths:")
    repeated = path_mapping[path_mapping["rows"] > 1]
    display(repeated.head(5))

print("\n" + "=" * 75)
print("STAGE 3B COMPLETED — SEND ME THE OUTPUT")
print("=" * 75)

PROJECT 19 — STAGE 3B: OUTPUT-PATH INVESTIGATION

========================= RAW MANIFEST =========================
Unique output paths: 480
Paths reused across folds, same window: 0
Paths associated with different windows: 480

Examples of path collisions:


,recordings,recording_window_pairs,folds,rows
output_path,,,,
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_10_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_11_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_12_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_13_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_14_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_15_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_16_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_17_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_18_raw_windows.npz,1,13,4,52



Sample repeated paths:


,recordings,recording_window_pairs,folds,rows
output_path,,,,
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_10_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_11_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_12_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_13_raw_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/raw_windows/N09_M07_F10_K001_14_raw_windows.npz,1,13,4,52



========================= ORDER MANIFEST =========================
Unique output paths: 480
Paths reused across folds, same window: 0
Paths associated with different windows: 480

Examples of path collisions:


,recordings,recording_window_pairs,folds,rows
output_path,,,,
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_10_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_11_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_12_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_13_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_14_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_15_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_16_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_17_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_18_windows.npz,1,13,4,52



Sample repeated paths:


,recordings,recording_window_pairs,folds,rows
output_path,,,,
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_10_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_11_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_12_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_13_windows.npz,1,13,4,52
/content/drive/MyDrive/BearingFaultResearch/data/processed/order_windows/K001/N09_M07_F10_K001_14_windows.npz,1,13,4,52



STAGE 3B COMPLETED — SEND ME THE OUTPUT


In [30]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 3C: CONFIRM EXPECTED OUTPUT-PATH REUSE
#
# Objective:
# Verify that each output path represents one recording,
# with its windows reused consistently across all four folds.
# ============================================================

import pandas as pd

print("=" * 75)
print("PROJECT 19 — STAGE 3C: EXPECTED OUTPUT-PATH REUSE")
print("=" * 75)

for name, df in [
    ("RAW", audit_data["raw_manifest"]),
    ("ORDER", audit_data["order_manifest"])
]:

    print(f"\n{'=' * 25} {name} MANIFEST {'=' * 25}")

    # Each output path should identify exactly one recording.
    path_summary = (
        df.groupby("output_path")
        .agg(
            recordings=("filename", "nunique"),
            windows=("window_index", "nunique"),
            folds=("fold", "nunique"),
            rows=("filename", "size")
        )
    )

    # Each recording/window pair should appear once in each fold.
    pair_fold_counts = (
        df.groupby(["output_path", "window_index", "fold"])
        .size()
    )

    # Expected: each output path maps to one recording,
    # 13 windows, 4 folds and 52 manifest rows.
    path_mapping_ok = (path_summary["recordings"] == 1).all()

    expected_rows_ok = (
        (path_summary["rows"] ==
         path_summary["windows"] * path_summary["folds"]).all()
    )

    all_four_folds_ok = (path_summary["folds"] == 4).all()

    pair_once_per_fold_ok = (pair_fold_counts == 1).all()

    # Check that all four folds occur for every recording/window pair.
    pair_fold_coverage = (
        df.groupby(["output_path", "window_index"])["fold"]
        .nunique()
    )

    all_windows_in_four_folds = (pair_fold_coverage == 4).all()

    print("Unique output paths:", len(path_summary))
    print("Unique recordings:", df["filename"].nunique())
    print("Minimum windows per path:", path_summary["windows"].min())
    print("Maximum windows per path:", path_summary["windows"].max())
    print("Minimum rows per path:", path_summary["rows"].min())
    print("Maximum rows per path:", path_summary["rows"].max())

    print("\nCHECKS:")
    print("One recording per output path:", path_mapping_ok)
    print("All four folds represented per path:", all_four_folds_ok)
    print("Rows equal windows × folds:", expected_rows_ok)
    print("Each window appears once per fold:", pair_once_per_fold_ok)
    print("Each window appears in all four folds:", all_windows_in_four_folds)

    print("\nRESULT:")
    if all([
        path_mapping_ok,
        all_four_folds_ok,
        expected_rows_ok,
        pair_once_per_fold_ok,
        all_windows_in_four_folds
    ]):
        print("PASS: Repeated output paths are consistent with expected fold reuse.")
    else:
        print("WARNING: Unexpected output-path pattern detected.")

print("\n" + "=" * 75)
print("STAGE 3C COMPLETED — SEND ME THE OUTPUT")
print("=" * 75)

PROJECT 19 — STAGE 3C: EXPECTED OUTPUT-PATH REUSE

========================= RAW MANIFEST =========================
Unique output paths: 480
Unique recordings: 480
Minimum windows per path: 13
Maximum windows per path: 24
Minimum rows per path: 52
Maximum rows per path: 96

CHECKS:
One recording per output path: True
All four folds represented per path: True
Rows equal windows × folds: True
Each window appears once per fold: True
Each window appears in all four folds: True

RESULT:
PASS: Repeated output paths are consistent with expected fold reuse.

========================= ORDER MANIFEST =========================
Unique output paths: 480
Unique recordings: 480
Minimum windows per path: 13
Maximum windows per path: 24
Minimum rows per path: 52
Maximum rows per path: 96

CHECKS:
One recording per output path: True
All four folds represented per path: True
Rows equal windows × folds: True
Each window appears once per fold: True
Each window appears in all four folds: True

RESULT:
PASS:

In [31]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 4: PREDICTION-TO-TEST-SET INTEGRITY
#
# Objective:
# Verify that predictions correspond exactly to the correct
# held-out test recordings, with consistent labels and models.
# ============================================================

import pandas as pd

split_df = audit_data["split_manifest"].copy()
pred_df = audit_data["predictions"].copy()

print("=" * 75)
print("PROJECT 19 — STAGE 4: PREDICTION INTEGRITY")
print("=" * 75)

# Expected test recordings from the split manifest
expected_test = split_df[split_df["split"] == "test"].copy()

# 1. Prediction row counts
print("\n1. PREDICTION COUNTS")

print("Total prediction rows:", len(pred_df))
print("Unique models:", pred_df["model"].nunique())
print("Models:", sorted(pred_df["model"].unique()))

print("\nPrediction rows per fold and model:")
display(
    pred_df.groupby(["fold", "model"]).size().unstack()
)

# 2. Check duplicate predictions
print("\n2. DUPLICATE PREDICTION CHECK")

duplicate_predictions = pred_df.duplicated(
    subset=["fold", "filename", "model"]
).sum()

print("Duplicate fold/recording/model predictions:", duplicate_predictions)

# 3. Compare predicted recordings against expected test recordings
print("\n3. TEST-SET MEMBERSHIP CHECK")

expected_keys = expected_test[["fold", "filename"]].drop_duplicates()
prediction_keys = pred_df[["fold", "filename"]].drop_duplicates()

membership = prediction_keys.merge(
    expected_keys,
    on=["fold", "filename"],
    how="outer",
    indicator=True
)

unexpected_predictions = (membership["_merge"] == "left_only").sum()
missing_test_recordings = (membership["_merge"] == "right_only").sum()

print("Predictions for non-test recordings:", unexpected_predictions)
print("Test recordings missing predictions:", missing_test_recordings)

# 4. Verify prediction labels against the split manifest
print("\n4. PREDICTION LABEL CONSISTENCY")

expected_labels = expected_test[
    ["fold", "filename", "label"]
].drop_duplicates()

label_check = pred_df.merge(
    expected_labels,
    on=["fold", "filename"],
    how="left",
    suffixes=("_prediction", "_manifest"),
    indicator=True
)

missing_manifest_matches = (label_check["_merge"] != "both").sum()

label_mismatches = (
    label_check["label_prediction"] != label_check["label_manifest"]
).sum()

print("Predictions without a matching test-manifest row:", missing_manifest_matches)
print("Prediction/manifest label mismatches:", label_mismatches)

# 5. Verify every model has the same test recordings
print("\n5. MODEL TEST-RECORDING CONSISTENCY")

model_recording_sets = {}

for model in sorted(pred_df["model"].unique()):
    model_recording_sets[model] = set(
        map(
            tuple,
            pred_df.loc[
                pred_df["model"] == model,
                ["fold", "filename"]
            ].drop_duplicates().to_numpy()
        )
    )

reference_model = sorted(model_recording_sets)[0]

for model, recordings in model_recording_sets.items():
    print(
        model,
        "| same test recordings as reference:",
        recordings == model_recording_sets[reference_model],
        "| unique fold/recording pairs:",
        len(recordings)
    )

# 6. Check expected number of predictions per test recording
print("\n6. PREDICTIONS PER TEST RECORDING")

prediction_counts = (
    pred_df.groupby(["fold", "filename"])["model"]
    .nunique()
)

print("Test recording/fold pairs:", len(prediction_counts))
print("Minimum models per test recording:", prediction_counts.min())
print("Maximum models per test recording:", prediction_counts.max())

# 7. Probability and predicted-label validity
print("\n7. PREDICTION VALUE CHECK")

probability_valid = pred_df["mean_probability"].between(0, 1).all()
predicted_labels_valid = pred_df["predicted_label"].isin([0, 1]).all()

print("All probabilities between 0 and 1:", probability_valid)
print("All predicted labels are 0 or 1:", predicted_labels_valid)

# 8. Summary
print("\n" + "=" * 75)
print("STAGE 4 SUMMARY")
print("=" * 75)

checks = {
    "No duplicate predictions": duplicate_predictions == 0,
    "No predictions outside the test set": unexpected_predictions == 0,
    "No missing test recordings": missing_test_recordings == 0,
    "All predictions match the test manifest": missing_manifest_matches == 0,
    "Prediction labels match manifest labels": label_mismatches == 0,
    "All models use identical test recordings": all(
        recordings == model_recording_sets[reference_model]
        for recordings in model_recording_sets.values()
    ),
    "Exactly three models predict each test recording": (
        prediction_counts.min() == 3 and prediction_counts.max() == 3
    ),
    "All probabilities are valid": probability_valid,
    "All predicted labels are valid": predicted_labels_valid
}

for check, passed in checks.items():
    print(("PASS: " if passed else "WARNING: ") + check)

print("\nSTAGE 4 COMPLETED — SEND ME THE FULL OUTPUT")

PROJECT 19 — STAGE 4: PREDICTION INTEGRITY

1. PREDICTION COUNTS
Total prediction rows: 1440
Unique models: 3
Models: ['Order CNN', 'Order CNN + Consistency', 'Raw CNN']

Prediction rows per fold and model:


model,Order CNN,Order CNN + Consistency,Raw CNN
fold,,,
1,120,120,120
2,120,120,120
3,120,120,120
4,120,120,120



2. DUPLICATE PREDICTION CHECK
Duplicate fold/recording/model predictions: 0

3. TEST-SET MEMBERSHIP CHECK
Predictions for non-test recordings: 0
Test recordings missing predictions: 0

4. PREDICTION LABEL CONSISTENCY
Predictions without a matching test-manifest row: 0
Prediction/manifest label mismatches: 0

5. MODEL TEST-RECORDING CONSISTENCY
Order CNN | same test recordings as reference: True | unique fold/recording pairs: 480
Order CNN + Consistency | same test recordings as reference: True | unique fold/recording pairs: 480
Raw CNN | same test recordings as reference: True | unique fold/recording pairs: 480

6. PREDICTIONS PER TEST RECORDING
Test recording/fold pairs: 480
Minimum models per test recording: 3
Maximum models per test recording: 3

7. PREDICTION VALUE CHECK
All probabilities between 0 and 1: True
All predicted labels are 0 or 1: True

STAGE 4 SUMMARY
PASS: No duplicate predictions
PASS: No predictions outside the test set
PASS: No missing test recordings
PASS: All pr

In [32]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 5: HELD-OUT OPERATING-CONDITION INTEGRITY
#
# Objective:
# Verify that each fold holds out the intended operating
# condition and excludes it from training and validation.
# ============================================================

import pandas as pd

split_df = audit_data["split_manifest"].copy()

# Extract operating condition from filename:
# Example: N09_M07_F10_K001_10.mat -> N09_M07_F10
split_df["condition_from_filename"] = (
    split_df["filename"]
    .str.extract(r"^(N\d+_M\d+_F\d+)_", expand=False)
)

print("=" * 75)
print("PROJECT 19 — STAGE 5: HELD-OUT CONDITION INTEGRITY")
print("=" * 75)

print("\n1. CONDITION PARSING CHECK")
print(
    "Recordings with unparsed condition:",
    split_df["condition_from_filename"].isna().sum()
)

print("\n2. HELD-OUT CONDITION PER FOLD")

heldout_counts = (
    split_df.groupby("fold")["held_out_condition"]
    .nunique()
)

display(heldout_counts)

print("\n3. ACTUAL TEST CONDITION VS DECLARED HELD-OUT CONDITION")

test_df = split_df[split_df["split"] == "test"].copy()

test_condition_comparison = (
    test_df.groupby(["fold", "held_out_condition"])
    ["condition_from_filename"]
    .agg(
        number_of_conditions="nunique",
        actual_conditions=lambda x: sorted(x.dropna().unique())
    )
    .reset_index()
)

display(test_condition_comparison)

# Check every test recording matches its declared held-out condition
test_matches = (
    test_df["condition_from_filename"]
    == test_df["held_out_condition"]
)

print(
    "Test recordings matching declared held-out condition:",
    int(test_matches.sum()),
    "out of",
    len(test_df)
)

print(
    "Test recordings with condition mismatch:",
    int((~test_matches).sum())
)

print("\n4. TRAINING AND VALIDATION EXCLUSION CHECK")

non_test_df = split_df[split_df["split"].isin(["train", "validation"])].copy()

non_test_matches = (
    non_test_df["condition_from_filename"]
    == non_test_df["held_out_condition"]
)

print(
    "Training/validation recordings matching held-out condition:",
    int(non_test_matches.sum())
)

print(
    "Training/validation recordings checked:",
    len(non_test_df)
)

print("\n5. CONDITIONS PRESENT IN EACH SPLIT")

condition_distribution = (
    split_df.groupby(["fold", "split"])["condition_from_filename"]
    .nunique()
    .unstack(fill_value=0)
)

display(condition_distribution)

print("\n6. FOLD CONDITION SUMMARY")

for fold in sorted(split_df["fold"].unique()):
    fold_df = split_df[split_df["fold"] == fold]

    declared = sorted(fold_df["held_out_condition"].dropna().unique())
    test_conditions = sorted(
        fold_df.loc[
            fold_df["split"] == "test",
            "condition_from_filename"
        ].dropna().unique()
    )

    train_conditions = sorted(
        fold_df.loc[
            fold_df["split"] == "train",
            "condition_from_filename"
        ].dropna().unique()
    )

    validation_conditions = sorted(
        fold_df.loc[
            fold_df["split"] == "validation",
            "condition_from_filename"
        ].dropna().unique()
    )

    print(f"\nFOLD {fold}")
    print("Declared held-out:", declared)
    print("Test conditions:", test_conditions)
    print("Training conditions:", train_conditions)
    print("Validation conditions:", validation_conditions)

# 7. SUMMARY
print("\n" + "=" * 75)
print("STAGE 5 SUMMARY")
print("=" * 75)

checks = {
    "All filenames have a parsed operating condition":
        split_df["condition_from_filename"].notna().all(),

    "Exactly one declared held-out condition per fold":
        (heldout_counts == 1).all(),

    "Every test recording matches its declared held-out condition":
        test_matches.all(),

    "No training/validation recording belongs to its fold's held-out condition":
        not non_test_matches.any()
}

for check, passed in checks.items():
    print(("PASS: " if passed else "WARNING: ") + check)

print("\nSTAGE 5 COMPLETED — SEND ME THE FULL OUTPUT")

PROJECT 19 — STAGE 5: HELD-OUT CONDITION INTEGRITY

1. CONDITION PARSING CHECK
Recordings with unparsed condition: 0

2. HELD-OUT CONDITION PER FOLD


,held_out_condition
fold,
1,1
2,1
3,1
4,1



3. ACTUAL TEST CONDITION VS DECLARED HELD-OUT CONDITION


,fold,held_out_condition,number_of_conditions,actual_conditions
0,1,N09_M07_F10,1,[N09_M07_F10]
1,2,N15_M01_F10,1,[N15_M01_F10]
2,3,N15_M07_F04,1,[N15_M07_F04]
3,4,N15_M07_F10,1,[N15_M07_F10]


Test recordings matching declared held-out condition: 480 out of 480
Test recordings with condition mismatch: 0

4. TRAINING AND VALIDATION EXCLUSION CHECK
Training/validation recordings matching held-out condition: 0
Training/validation recordings checked: 1440

5. CONDITIONS PRESENT IN EACH SPLIT


split,test,train,validation
fold,,,
1,1,3,3
2,1,3,3
3,1,3,3
4,1,3,3



6. FOLD CONDITION SUMMARY

FOLD 1
Declared held-out: ['N09_M07_F10']
Test conditions: ['N09_M07_F10']
Training conditions: ['N15_M01_F10', 'N15_M07_F04', 'N15_M07_F10']
Validation conditions: ['N15_M01_F10', 'N15_M07_F04', 'N15_M07_F10']

FOLD 2
Declared held-out: ['N15_M01_F10']
Test conditions: ['N15_M01_F10']
Training conditions: ['N09_M07_F10', 'N15_M07_F04', 'N15_M07_F10']
Validation conditions: ['N09_M07_F10', 'N15_M07_F04', 'N15_M07_F10']

FOLD 3
Declared held-out: ['N15_M07_F04']
Test conditions: ['N15_M07_F04']
Training conditions: ['N09_M07_F10', 'N15_M01_F10', 'N15_M07_F10']
Validation conditions: ['N09_M07_F10', 'N15_M01_F10', 'N15_M07_F10']

FOLD 4
Declared held-out: ['N15_M07_F10']
Test conditions: ['N15_M07_F10']
Training conditions: ['N09_M07_F10', 'N15_M01_F10', 'N15_M07_F04']
Validation conditions: ['N09_M07_F10', 'N15_M01_F10', 'N15_M07_F04']

STAGE 5 SUMMARY
PASS: All filenames have a parsed operating condition
PASS: Exactly one declared held-out condition per fold

In [33]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 6A: NORMALIZATION AND PREPROCESSING ARTIFACT INVENTORY
#
# Objective:
# Locate saved normalization statistics and preprocessing
# configuration files for all four folds.
# No files are modified.
# ============================================================

import os

root = "/content/drive/MyDrive/BearingFaultResearch"

search_roots = [
    os.path.join(root, "fresh_train_predict_run"),
    os.path.join(root, "results"),
    os.path.join(root, "models")
]

keywords = [
    "norm",
    "scaler",
    "mean",
    "std",
    "standard",
    "preprocess",
    "config"
]

print("=" * 75)
print("PROJECT 19 — STAGE 6A: PREPROCESSING ARTIFACT INVENTORY")
print("=" * 75)

found_files = []

for search_root in search_roots:
    if not os.path.exists(search_root):
        print("\nFolder not found:", search_root)
        continue

    for current_root, dirs, files in os.walk(search_root):
        for filename in files:
            lower_name = filename.lower()

            if any(keyword in lower_name for keyword in keywords):
                full_path = os.path.join(current_root, filename)
                found_files.append(full_path)

print("\nMatching files found:", len(found_files))

for path in sorted(set(found_files)):
    print(path)

print("\n" + "=" * 75)
print("STAGE 6A COMPLETED")
print("Send me the complete output before proceeding.")
print("=" * 75)

PROJECT 19 — STAGE 6A: PREPROCESSING ARTIFACT INVENTORY

Matching files found: 14
/content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fold1_consistency_normalization.json
/content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fold1_order_normalization.json
/content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fold1_raw_normalization.json
/content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fold2_consistency_normalization.json
/content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fold2_order_normalization.json
/content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fold2_raw_normalization.json
/content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fold3_consistency_normalization.json
/content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/fold3_order_normalization.json
/content/drive/MyDrive/BearingFaultResearch/fres

In [34]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 6B: INSPECT NORMALIZATION STATISTICS
#
# Objective:
# Read the saved normalization parameters for all models
# and folds, and identify the fields available for auditing.
# ============================================================

import os
import json

results_dir = (
    "/content/drive/MyDrive/BearingFaultResearch/"
    "fresh_train_predict_run/results"
)

print("=" * 75)
print("PROJECT 19 — STAGE 6B: NORMALIZATION STATISTICS")
print("=" * 75)

normalization_files = sorted([
    f for f in os.listdir(results_dir)
    if f.endswith("_normalization.json")
])

print("\nNormalization files found:", len(normalization_files))

for filename in normalization_files:
    path = os.path.join(results_dir, filename)

    print("\n" + "-" * 75)
    print("FILE:", filename)
    print("-" * 75)

    with open(path, "r") as file:
        data = json.load(file)

    print(json.dumps(data, indent=2))

print("\n" + "=" * 75)
print("STAGE 6B COMPLETED — SEND ME THE FULL OUTPUT")
print("=" * 75)

PROJECT 19 — STAGE 6B: NORMALIZATION STATISTICS

Normalization files found: 12

---------------------------------------------------------------------------
FILE: fold1_consistency_normalization.json
---------------------------------------------------------------------------
{
  "mean": 0.004973001078062809,
  "std": 0.32441127962602123
}

---------------------------------------------------------------------------
FILE: fold1_order_normalization.json
---------------------------------------------------------------------------
{
  "mean": 0.004973001078062809,
  "std": 0.32441127962602123
}

---------------------------------------------------------------------------
FILE: fold1_raw_normalization.json
---------------------------------------------------------------------------
{
  "mean": 0.005046434261576848,
  "std": 0.3247729544682251
}

---------------------------------------------------------------------------
FILE: fold2_consistency_normalization.json
---------------------------------

In [36]:
# ============================================================
# PROJECT 19 — FINAL RESEARCH-INTEGRITY AUDIT
# STAGE 6C: TRACE NORMALIZATION CALCULATION
#
# Objective:
# Find the source-code locations where normalization
# statistics are calculated and saved.
# ============================================================

import os

root = "/content/drive/MyDrive/BearingFaultResearch"

keywords = [
    "normalization.json",
    "np.mean",
    "np.std",
    "mean(",
    "std(",
    "normalization_mean",
    "normalization_std"
]

print("=" * 75)
print("PROJECT 19 — STAGE 6C: NORMALIZATION SOURCE-CODE SEARCH")
print("=" * 75)

matches = []

for current_root, dirs, files in os.walk(root):
    # Avoid searching generated data and model artifacts
    dirs[:] = [
        d for d in dirs
        if d not in [
            ".git", "__pycache__", ".venv",
            "raw", "processed", "checkpoints"
        ]
    ]

    for filename in files:
        if not filename.endswith(".py"):
            continue

        path = os.path.join(current_root, filename)

        try:
            with open(path, "r", encoding="utf-8") as file:
                lines = file.readlines()
        except Exception:
            continue

        for index, line in enumerate(lines):
            if any(keyword in line.lower() for keyword in keywords):
                matches.append((path, index, lines))

print("\nMatching source-code locations:", len(matches))

shown = set()

for path, index, lines in matches:
    key = (path, index)

    if key in shown:
        continue

    shown.add(key)

    print("\n" + "-" * 75)
    print("FILE:", path)
    print("LINE:", index + 1)
    print("-" * 75)

    start = max(0, index - 4)
    end = min(len(lines), index + 6)

    for line_number in range(start, end):
        print(f"{line_number + 1:4}: {lines[line_number].rstrip()}")

print("\n" + "=" * 75)
print("STAGE 6C COMPLETED — SEND ME THE OUTPUT")
print("=" * 75)

PROJECT 19 — STAGE 6C: NORMALIZATION SOURCE-CODE SEARCH

Matching source-code locations: 0

STAGE 6C COMPLETED — SEND ME THE OUTPUT


In [39]:
# ==========================================================
# PROJECT 19 — STAGE 1: PROJECT LOCATION AUDIT
# Objective: locate the project folder and identify its
# configured dataset and processed-data directories.
# ==========================================================

from pathlib import Path

candidates = [
    Path(r"C:\Users\user\PycharmProjects\Project19_Order_Informed_Domain_Generalization"),
    Path(r"C:\Users\user\Desktop\BearingFaultResearch"),
]

print("=" * 70)
print("PROJECT 19 — PROJECT LOCATION AUDIT")
print("=" * 70)

for root in candidates:
    print("\nProject path:", root)
    print("Exists:", root.exists())

    if root.exists():
        print("Contains:")
        for item in list(root.iterdir())[:20]:
            print("  ", item.name)

PROJECT 19 — PROJECT LOCATION AUDIT

Project path: C:\Users\user\PycharmProjects\Project19_Order_Informed_Domain_Generalization
Exists: False

Project path: C:\Users\user\Desktop\BearingFaultResearch
Exists: False


In [40]:
# PROJECT 19 — FIND SAVED CSV RESULTS

from pathlib import Path

search_locations = [
    Path("/content"),
    Path("/content/drive/MyDrive")
]

keywords = ["fold_metrics", "predictions"]

print("SEARCHING FOR PROJECT 19 CSV FILES")
print("=" * 50)

for root in search_locations:
    if root.exists():
        for file in root.rglob("*.csv"):
            if any(word in file.name.lower() for word in keywords):
                print("FOUND:", file)

SEARCHING FOR PROJECT 19 CSV FILES
FOUND: /content/drive/MyDrive/BearingFaultResearch/results/stage8_raw_cnn_recording_level_predictions.csv
FOUND: /content/drive/MyDrive/BearingFaultResearch/results/stage9_order_cnn_recording_level_predictions.csv
FOUND: /content/drive/MyDrive/BearingFaultResearch/results/stage10_fold1_consistency_recording_predictions.csv
FOUND: /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/all_fold_metrics.csv
FOUND: /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/predictions/all_recording_level_predictions.csv
FOUND: /content/drive/MyDrive/BearingFaultResearch/results/stage8_raw_cnn_recording_level_predictions.csv
FOUND: /content/drive/MyDrive/BearingFaultResearch/results/stage9_order_cnn_recording_level_predictions.csv
FOUND: /content/drive/MyDrive/BearingFaultResearch/results/stage10_fold1_consistency_recording_predictions.csv
FOUND: /content/drive/MyDrive/BearingFaultResearch/fresh_train_predict_run/results/all_f

In [44]:
# PROJECT 19 — FIND EXPERIMENT FIGURES

from pathlib import Path

root = Path("/content/drive/MyDrive/BearingFaultResearch")

for folder in root.rglob("*"):
    if folder.is_file() and folder.suffix.lower() in [".png", ".jpg", ".jpeg"]:
        print(folder)

## Interpretation reminder

- The four folds represent **held-out operating conditions**, not four unseen-speed tests.
- Fold 1 is the direct unseen-speed evaluation: its held-out N09 condition has a mean speed around 900 RPM, while the training recordings are around 1,500 RPM.
- Fold 2 holds out the M01 factor, Fold 3 holds out F04, and Fold 4 holds out a combination of otherwise familiar factors.
- Report the four-fold mean and standard deviation, and discuss Fold 1 separately as the direct unseen-speed result.
- The consistency model is trained and evaluated on all four folds in this fresh run, making the three-model comparison more complete than the earlier Fold-1-only consistency experiment.
